# **1. DATA EXPLORE**

**Các thư viện cần thiết**

In [128]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import re
import unicodedata
from sklearn.impute import KNNImputer
import warnings
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')

## **1.1. Kiểu dữ liệu**

**In ra kiểu dữ liệu để kiểm tra**

In [ ]:
df = pd.read_csv("data.csv")
df.head()

# Lấy kiểu dữ liệu
df.dtypes
df.describe()
df.info()

# Tạo bản sao để xử lý
df_typed = df.copy()

**Xử lý các cột chứa kí tự "%" (xóa "%" và đưa về dạng số)**

In [130]:
# Xử lý cột 'cancel_by_seller_rate' và 'return_rate'
percent_columns = ['cancel_by_seller_rate', 'return_rate']

for col in percent_columns:
    if col in df_typed.columns:
        # Đếm số lượng giá trị không phải NaN
        non_null_count = df_typed[col].notna().sum()

        if non_null_count > 0:
            # Kiểm tra xem có chứa ký tự '%' không
            has_percent = df_typed[col].astype(str).str.contains('%').any()

            # Xử lý chuyển đổi
            if has_percent:
                # Loại bỏ ký tự '%' và chuyển đổi sang số
                df_typed[col] = df_typed[col].astype(str).str.replace('%', '', regex=False)

                # Loại bỏ khoảng trắng
                df_typed[col] = df_typed[col].str.strip()

                # Chuyển đổi sang số, không hợp lệ sẽ thành NaN
                df_typed[col] = pd.to_numeric(df_typed[col], errors='coerce')

                # Chia cho 100 để chuyển từ phần trăm sang số thập phân
                df_typed[col] = df_typed[col] / 100
            else:
                # Nếu không có '%', thử chuyển trực tiếp sang số
                df_typed[col] = pd.to_numeric(df_typed[col], errors='coerce')
            
        df_typed[col] = df_typed[col].astype('float64')
    else:
        print(f"\nCột '{col}' không tồn tại trong dataset")

**Đưa các cột số về đúng kiểu dữ liệu của nó**

In [ ]:
# Danh sách các cột số cần kiểm tra
numeric_columns_to_check = [
    'price', 'original_price', 'discount_rate', 'quantity_sold',
    'rating_average', 'review_count', 'is_return_policy',
    'is_freeship_xtra', 'is_authentic', 'image_count', 'video_count',
    'is_brand', 'store_review_count', 'total_follower', 'is_official',
]

for col in numeric_columns_to_check:
    if col in df_typed.columns:
        if df_typed[col].dtype == 'object':
            print(f"\nCột '{col}' có kiểu object, đang chuyển sang numeric...")

            # Đếm số giá trị không phải NaN trước khi chuyển
            before_count = df_typed[col].notna().sum()

            # Chuyển đổi sang numeric
            df_typed[col] = pd.to_numeric(df_typed[col], errors='coerce')
            
            # Đếm số giá trị không phải NaN sau khi chuyển
            after_count = df_typed[col].notna().sum()

            lost_values = before_count - after_count
            if lost_values > 0:
                print(f"Mất {lost_values} giá trị không thể chuyển đổi")

            print(f"Kiểu dữ liệu sau: {df_typed[col].dtype}")
        
        df_typed[col] = df_typed[col].astype('float64')

# Lưu file CSV mới
output_file_path = 'data_typed.csv'
df_typed.to_csv(output_file_path, index=False)
print(f"\nĐã lưu file đã xử lý tại: {output_file_path}")

**In ra kiểu dữ liệu của các cột để kiểm tra**

In [ ]:
df_typed.dtypes
df_typed.info()

## **1.2. Khám phá Missing Value dạng text**

In [ ]:
df = pd.read_csv("data_typed.csv")

# Các cột text
text_cols = [
    'product_name',
    'product_url',
    'category_name',
    'category_root_name',
    'brand_name',
    'origin',
    'store_name',
    'cancel_by_seller_rate_status',
    'return_rate_status'
]

# Dictionary để lưu cột và số dòng bị thiếu ở cột đó
missing_dict = {}

count = 0
for col in text_cols:
    number_missing_value = df[col].isnull().sum()
    if(number_missing_value>0): 
        count+=1
    missing_dict.update({col:number_missing_value})

print(f"{'Cột':30} {'Số lượng':10} {'Phần trăm':10}")
print("-" * 55)

for col, val in missing_dict.items():
    percent = round(val / len(df) * 100, 3)
    print(f"{col:30} {val:<10} {percent:<10}%")

print("\nSố cột chứa missing value: ", count)

## **1.3. Khám phá Missing Value dạng numberic**

**Thống kê tỉ lệ Missing Value của các cột numberic**

In [ ]:
# Danh sách các cột cần kiểm tra
columns_to_check = [
    'product_id', 'category_id', 'price', 'original_price', 
    'discount_rate', 'quantity_sold', 'rating_average', 
    'review_count', 'is_return_policy', 'is_freeship_xtra', 
    'is_authentic', 'image_count', 'video_count', 'is_brand', 
    'store_id', 'store_review_count', 'total_follower', 
    'is_official', 'cancel_by_seller_rate', 'return_rate'
]

# Kiểm tra xem các cột có tồn tại trong dataframe không
existing_columns = [col for col in columns_to_check if col in df.columns]
missing_columns = [col for col in columns_to_check if col not in df.columns]

if missing_columns:
    print(f"Các cột không tồn tại trong dataset: {missing_columns}")
    print(f"Tiếp tục với {len(existing_columns)} cột có sẵn...")

# Tính missing values
total_rows = len(df)

missing_data = []
for col in existing_columns:
    missing_count = df[col].isnull().sum()
    missing_percent = (missing_count / total_rows) * 100

    missing_data.append({
        'Thuộc tính': col,
        'Số lượng missing': missing_count,
        'Tỉ lệ missing (%)': round(missing_percent, 2)
    })

# Tạo dataframe missing_df
missing_df = pd.DataFrame(missing_data)

print("\nBẢNG TỈ LỆ MISSING VALUES:")
print(missing_df.to_string(index=False))

**Trực quan hóa kết quả tỉ lệ Missing Value trên**

In [ ]:
# Trực quan hóa kết quả
# Tạo biểu đồ tỉ lệ missing values
plt.figure(figsize=(12, 6))
bars = plt.bar(missing_df['Thuộc tính'], missing_df['Tỉ lệ missing (%)'], color='skyblue')
plt.axhline(y=25, color='r', linestyle='--', alpha=0.7, label='Ngưỡng 25%')
plt.xlabel('Thuộc tính')
plt.ylabel('Tỉ lệ Missing Values (%)')
plt.title('TỈ LỆ MISSING VALUES THEO TỪNG THUỘC TÍNH')
plt.xticks(rotation=45, ha='right')
plt.legend()
plt.tight_layout()

# Tô màu đỏ cho các cột có missing > 25%
for i, (bar, rate) in enumerate(zip(bars, missing_df['Tỉ lệ missing (%)'])):
    if rate > 25:
        bar.set_color('red')

plt.show()

**Bảng tổng hợp chi tiết về phân bổ của các cột**

In [ ]:
# Hiển thị thông tin chi tiết về phân bố của TẤT CẢ các cột dưới dạng BẢNG
print("\nTHÔNG TIN CHI TIẾT VỀ PHÂN BỐ CỦA TẤT CẢ CÁC CỘT (BẢNG TỔNG HỢP):")
print("=" * 80)

summary_stats = []

for col in existing_columns:
    col_data = df[col]
    dtype = col_data.dtype
    n_unique = col_data.nunique()
    missing_count = col_data.isnull().sum()
    missing_percentage = (missing_count / total_rows) * 100

    # Giá trị mặc định cho các thống kê số
    min_val = max_val = mean_val = median_val = std_val = var_val = q25 = q75 = np.nan

    if pd.api.types.is_numeric_dtype(col_data):
        numeric_data = col_data.dropna()
        if len(numeric_data) > 0:
            min_val = numeric_data.min()
            max_val = numeric_data.max()
            mean_val = numeric_data.mean()
            median_val = numeric_data.median()
            std_val = numeric_data.std()
            var_val = numeric_data.var()
            q25 = numeric_data.quantile(0.25)
            q75 = numeric_data.quantile(0.75)

    summary_stats.append({
        'Thuộc tính': col,
        'Kiểu dữ liệu': dtype,
        'Số giá trị unique': n_unique,
        'Số lượng missing': missing_count,
        'Tỉ lệ missing (%)': round(missing_percentage, 2),
        'Min': min_val,
        'Max': max_val,
        'Mean': mean_val,
        'Median': median_val,
        'Std': std_val,
        'Var': var_val,
        'Q1 (25%)': q25,
        'Q3 (75%)': q75
    })

summary_df = pd.DataFrame(summary_stats)

# Sắp xếp cột cho đẹp (có thể chỉnh lại thứ tự nếu muốn)
cols_order = [
    'Thuộc tính', 'Kiểu dữ liệu', 'Số giá trị unique',
    'Số lượng missing', 'Tỉ lệ missing (%)',
    'Min', 'Max', 'Mean', 'Median', 'Std', 'Var',
    'Q1 (25%)', 'Q3 (75%)'
]
summary_df = summary_df[cols_order]

# In bảng thống kê
print(summary_df.to_string(index=False))
print("=" * 80)

## **1.4. Thống kê trùng lặp (theo `product_id` và `product_url`)**

In [ ]:
# Kiểm tra để đảm bảo không có giá trị trùng lặp hoặc không hợp lệ
product_cols = ['product_id', 'product_url']

for col in product_cols:
    duplicates = df[col][df[col].duplicated()]
    if not duplicates.empty:
        print(f"Có giá trị trùng trong cột '{col}':")
        print(duplicates.unique())
    else:
        print(f"Không có giá trị trùng trong cột '{col}':")

## **1.5. Khám phá Outliers**

**Phân tích và thống kê Outliers**

In [ ]:
df_outlier = df

# DANH SÁCH CÁC CỘT CẦN PHÂN TÍCH
columns_for_boxplot = [
    'price', 'original_price', 'discount_rate', 'quantity_sold',
    'rating_average', 'review_count', 'image_count', 'video_count',
    'store_review_count', 'total_follower', 
    'cancel_by_seller_rate', 'return_rate'
]

# TÍNH TOÁN THÔNG SỐ OUTLIERS
print("="*80)
print("PHÂN TÍCH OUTLIERS THEO PHƯƠNG PHÁP IQR")
print("="*80)

results = []
for col in columns_for_boxplot:
    if col in df_outlier.columns:
        data = df_outlier[col]
        
        # Tính IQR và bounds
        Q1 = data.quantile(0.25)
        Q3 = data.quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        # Xác định outliers
        outliers = data[(data < lower_bound) | (data > upper_bound)]
        
        # Tính skewness (độ lệch)
        median = data.median()
        mean = data.mean()
        
            # Nhận xét về phân phối
        if median < mean:
            skewness_note = "Lệch phải (right-skewed)"
        elif median > mean:
            skewness_note = "Lệch trái (left-skewed)"
        else:
            skewness_note = "Cân bằng"
        # Lưu kết quả
        results.append({
            'Cột': col,
            'IQR': round(IQR, 2),
            'Lower Bound': round(lower_bound, 2),
            'Upper Bound': round(upper_bound, 2),
            'Số outliers': len(outliers),
            'Tỉ lệ outliers': f"{len(outliers)/len(data)*100:.1f}%",
            'Min outlier': round(outliers.min(), 2) if len(outliers) > 0 else "Không có",
            'Max outlier': round(outliers.max(), 2) if len(outliers) > 0 else "Không có",
            'Phân phối': skewness_note
        })

# IN BẢNG KẾT QUẢ
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

**Vẽ Box Plot**

In [ ]:
# VẼ BOX PLOT
print("\n" + "="*80)
print("VẼ BOX PLOT")
print("="*80)

# Tạo figure lớn với nhiều subplot
fig, axes = plt.subplots(3, 4, figsize=(20, 15))
axes = axes.flatten()

for i, col in enumerate(columns_for_boxplot):
    if i < len(axes):
        ax = axes[i]
        
        # Vẽ box plot
        boxplot = ax.boxplot(df[col].dropna(), patch_artist=True)
        
        # Tùy chỉnh màu sắc 
        boxplot['boxes'][0].set_facecolor('lightblue')
        boxplot['medians'][0].set_color('red')
        
        # Thêm thông tin
        median_val = df[col].median()
        outliers_count = results[i]['Số outliers']
        
        # Đặt tiêu đề với thông tin phân phối
        color = 'red' if 'Lệch phải' in results[i]['Phân phối'] else 'blue' if 'Lệch trái' in results[i]['Phân phối'] else 'green'
        ax.set_title(f"{col}\n({results[i]['Phân phối']})", 
                    fontsize=12, fontweight='bold', color=color)
        ax.set_ylabel('Giá trị')
        ax.grid(True, alpha=0.3)
        
        # Hiển thị số outliers
        ax.text(0.95, 0.95, f'Outliers: {outliers_count}', 
                transform=ax.transAxes, fontsize=10,
                verticalalignment='top', horizontalalignment='right',
                bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))

# Ẩn các subplot không sử dụng
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('BOX PLOT - PHÂN TÍCH OUTLIERS VÀ PHÂN PHỐI', 
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## **1.6. Data Reduction sơ bộ**

### **1.6.1. Numberic**

In [ ]:
# Danh sách cột numberic
numeric_cols = [
    'product_id', 'category_id', 'price',
    'original_price', 'discount_rate', 'quantity_sold',
    'rating_average', 'review_count', 'is_return_policy',
    'is_freeship_xtra', 'is_authentic','image_count', 
    'video_count', 'is_brand', 'store_id', 'store_review_count', 
    'total_follower', 'is_official', 'cancel_by_seller_rate', 'return_rate',
]

df_pre_data_reduction = df
# --- THỐNG KÊ CỘT SỐ (NUMERIC) ---
print("--- TÓM TẮT CỘT SỐ (FLOAT64) ---")

for col in numeric_cols:
    if col in df_pre_data_reduction.columns:
        print(f"\n[Cột: {col}]")
        
        # Tính toán thống kê
        unique_count = df_pre_data_reduction[col].nunique(dropna=True)
        stats = df_pre_data_reduction[col].describe(percentiles=[]).to_dict()
        missing_count = df_pre_data_reduction[col].isna().sum()
        
        print(f"  > Số lượng giá trị khác nhau (không kể NaN): {int(unique_count)}")
        print(f"  > Giá trị thiếu (NaN): {missing_count} ({(missing_count/total_rows)*100:.2f}%)")
        print(f"  > Trung bình (Mean): {stats.get('mean', 'N/A'):.4f}")
        print(f"  > Độ lệch chuẩn (Std): {stats.get('std', 'N/A'):.4f}")
        print(f"  > Min/Max: {stats.get('min', 'N/A')} / {stats.get('max', 'N/A')}")
    
print("\n" + "="*50 + "\n")

### **1.6.2. Text**

In [ ]:
# Danh sách cột text
text_cols = [
    'product_name',
    'product_url',
    'category_name',
    'category_root_name',
    'brand_name',
    'origin',
    'store_name',
    'cancel_by_seller_rate_status',
    'return_rate_status'
]

# --- THỐNG KÊ CỘT CHUỖI (TEXT/OBJECT) ---
print("--- TÓM TẮT CỘT CHUỖI (TEXT/OBJECT) ---")

for col in text_cols:
    if col in df_pre_data_reduction.columns:
        print(f"\n[Cột: {col}]")
        
        unique_count = df_pre_data_reduction[col].nunique(dropna=True)
        missing_count = df_pre_data_reduction[col].isna().sum()
        value_percentages = df_pre_data_reduction[col].value_counts(dropna=False, normalize=True) * 100
        
        print(f"  > Số lượng giá trị khác nhau (không kể NaN): {unique_count}")
        print(f"  > Giá trị thiếu (NaN): {missing_count} ({(missing_count/total_rows)*100:.2f}%)")

## **1.7. EDA**

# **2. DATA PREPROCESSING**

## **2.1. Data Cleaning**

### **2.1.1. Xử lý Missing Value dạng numberic**

**Đọc dữ liệu và xác định các cột cần xử lý**

In [142]:
df = pd.read_csv("data_typed.csv")

# Danh sách các cột cần xử lý
columns_to_check = [
    'quantity_sold', 'is_freeship_xtra', 'is_authentic', 
    'store_id', 'store_review_count', 'total_follower', 
    'is_official', 'cancel_by_seller_rate', 'return_rate'
]

# Chỉ lấy các cột có trong dataframe
existing_columns = [col for col in columns_to_check if col in df.columns]
missing_columns = [col for col in columns_to_check if col not in df.columns]

if missing_columns:
    print(f"Các cột không tồn tại trong dataset: {missing_columns}")
    print(f"Tiếp tục với {len(existing_columns)} cột có sẵn...")


# Tạo bản sao để xử lý
df_imputed = df.copy()

**Điền 0 cho các cột `quantity_sold`, `is_`, `store_id` bị Missing Value**

In [ ]:
# Xác định các cột cần điền 0.0
zero_impute_cols = []

# 1. quantity_sold
if 'quantity_sold' in existing_columns:
    zero_impute_cols.append('quantity_sold')
    missing_count = df_imputed['quantity_sold'].isnull().sum()
    if missing_count > 0:
        df_imputed['quantity_sold'] = df_imputed['quantity_sold'].fillna(0.0)
        print(f"Đã điền 0.0 cho quantity_sold: {missing_count} giá trị")

# 2. Các cột bắt đầu bằng 'is_'
is_cols = [col for col in existing_columns if col.startswith('is_')]
for col in is_cols:
    zero_impute_cols.append(col)
    missing_count = df_imputed[col].isnull().sum()
    if missing_count > 0:
        df_imputed[col] = df_imputed[col].fillna(0.0)
        print(f"Đã điền 0.0 cho {col}: {missing_count} giá trị")

# 3. store_id
if 'store_id' in existing_columns:
    zero_impute_cols.append('store_id')
    missing_count = df_imputed['store_id'].isnull().sum()
    if missing_count > 0:
        df_imputed['store_id'] = df_imputed['store_id'].fillna(0.0)
        print(f"Đã điền 0.0 cho store_id: {missing_count} giá trị")

**Dùng KNN Imputer để điền Missing Value cho các cột còn lại**

In [ ]:
# CHUẨN BỊ DỮ LIỆU CHO KNN IMPUTER
# Xác định các cột còn lại cần dùng KNN (các cột số không phải là cột đặc biệt)
remaining_numeric_cols = []
for col in existing_columns:
    if col not in zero_impute_cols and pd.api.types.is_numeric_dtype(df_imputed[col]):
        remaining_numeric_cols.append(col)

print(f"Các cột sẽ dùng KNN Imputer ({len(remaining_numeric_cols)} cột):")
for i, col in enumerate(remaining_numeric_cols, 1):
    missing_count = df_imputed[col].isnull().sum()
    print(f"  {i}. {col}: {missing_count} missing")

# Kiểm tra xem còn missing values không
missing_after_zero = {}

for col in remaining_numeric_cols:
    missing_count = df_imputed[col].isnull().sum()

    if missing_count > 0:
        missing_after_zero[col] = missing_count

# ÁP DỤNG KNN IMPUTER
if missing_after_zero:
    # Tạo subset dữ liệu chỉ chứa các cột cần impute
    knn_data = df_imputed[remaining_numeric_cols].copy()
    
    # Kiểm tra xem có đủ dữ liệu để thực hiện KNN không
    min_non_missing = knn_data.notna().sum().min()
    if min_non_missing < 2:
        print("Cảnh báo: Một số cột có quá ít dữ liệu không bị missing (<2 giá trị)")
        print("KNN có thể không hoạt động hiệu quả.")
    
    # Áp dụng KNN Imputer
    # Tùy chọn 1: KNN với tham số distance ưu tiên các điểm gần hơn
    knn_imputer = KNNImputer(n_neighbors=5, weights='distance')

    # Thực hiện imputation
    knn_imputed_array = knn_imputer.fit_transform(knn_data)
    
    # Cập nhật dữ liệu đã impute vào dataframe
    knn_imputed_df = pd.DataFrame(knn_imputed_array, 
                                  columns=remaining_numeric_cols, 
                                  index=df_imputed.index)
    
    # Cập nhật các cột đã impute
    for col in remaining_numeric_cols:
        df_imputed[col] = knn_imputed_df[col]
    
# LƯU FILE ĐÃ IMPUTE
output_file_path = 'data_imputed.csv'
df_imputed.to_csv(output_file_path, index=False)
print(f" Đã lưu file đã impute tại: {output_file_path}")

**Thống kê sau khi impute**

In [ ]:
# PHÂN TÍCH THỐNG KÊ SAU KHI IMPUTE (BẢNG 1 DÒNG / THUỘC TÍNH)
print("\n" + "="*60)
print("BƯỚC 5: THỐNG KÊ TRƯỚC & SAU KHI XỬ LÍ MISSING VALUES:")
print("="*60)

important_cols = [
    'quantity_sold',
    'is_freeship_xtra', 'is_authentic',
    'store_review_count', 'total_follower', 'is_official',
    'cancel_by_seller_rate', 'return_rate'
]

important_cols = [col for col in important_cols if col in df_imputed.columns]

rows = []

for col in important_cols:
    original_data = df[col].dropna()
    imputed_data = df_imputed[col]

    # BEFORE stats (nếu toàn NaN → điền NaN)
    if len(original_data) == 0:
        before = {k: np.nan for k in ['min','max','mean','median','std','q1','q3']}
        mean_change = np.nan
    else:
        before = {
            'min': original_data.min(),
            'max': original_data.max(),
            'mean': original_data.mean(),
            'median': original_data.median(),
            'std': original_data.std(),
            'q1': original_data.quantile(0.25),
            'q3': original_data.quantile(0.75)
        }
        mean_change = ((imputed_data.mean() - before['mean']) / before['mean']) * 100

    # AFTER stats
    after = {
        'min': imputed_data.min(),
        'max': imputed_data.max(),
        'mean': imputed_data.mean(),
        'median': imputed_data.median(),
        'std': imputed_data.std(),
        'q1': imputed_data.quantile(0.25),
        'q3': imputed_data.quantile(0.75),
    }

    # Thêm 1 dòng duy nhất per feature
    rows.append({
        'Thuộc tính': col,
        'Min_before': before['min'],
        'Min_after': after['min'],
        'Max_before': before['max'],
        'Max_after': after['max'],
        'Mean_before': before['mean'],
        'Mean_after': after['mean'],
        'Median_before': before['median'],
        'Median_after': after['median'],
        'Std_before': before['std'],
        'Std_after': after['std'],
        'Q1_before': before['q1'],
        'Q1_after': after['q1'],
        'Q3_before': before['q3'],
        'Q3_after': after['q3'],
        'Mean_change(%)': mean_change
    })

summary_df = pd.DataFrame(rows)

# Làm gọn số
numeric_cols = summary_df.select_dtypes(include=[np.number]).columns
summary_df[numeric_cols] = summary_df[numeric_cols].round(4)

print("\nBẢNG THỐNG KÊ TRƯỚC – SAU KHi XỬ LÍ MISSING VALUES:")
print(summary_df.to_string(index=False))
print("-" * 80)

### **2.1.2. Xử lý Missing Value dạng text**

**Đa số điền "Unknown" cho các cột text bị Missing Value, riêng `store_name` thì dựa trên `store_id` để mapping nếu tìm thấy trong bộ dữ liệu, nếu không cũng điền "Unknown"**

In [ ]:
df = pd.read_csv("data_imputed.csv")

# Đối với brand_name và origin điền "Unknown"
print("Số lượng missing value trước khi điền: ")
for col in ['brand_name','origin']:
    print(f"{col}: {missing_dict[col]}")
    df[col] = df[col].fillna('Unknown')

# Kiểm tra và cập nhật missing_dict
print("Số lượng missing value sau khi điền: ")
for col in ['brand_name','origin']:
    number_missing_value = df[col].isna().sum()
    missing_dict[col] = number_missing_value
    
    print(f"{col}: {missing_dict[col]}")     
    
# Đối với store_name, lấy store_id và đối chiếu với các hàng khác để tìm store_name
# Nếu không có thì điền "Unknown"
print("Số lượng missing value trước khi điền: ")
for col in ['store_name']:
    print(f"{col}: {missing_dict[col]}")
    
# Tạo dictionary mapping store_id -> store_name
store_map = df.dropna(subset=['store_name']).drop_duplicates('store_id')
store_map = store_map.set_index('store_id')['store_name'].to_dict()

def fill_store_name(row):
    if pd.isna(row['store_name']):
        return store_map.get(row['store_id'], "Unknown")
    return row['store_name']

df['store_name'] = df.apply(fill_store_name, axis =1)

# Thống kê số lượng "Unknown"
unknown_count = (df['store_name'] == "Unknown").sum()
unknown_percent = round(unknown_count / missing_dict['store_name'] * 100, 3)
print(f"\nTỷ lệ Unknown trong {missing_dict['store_name']} giá trị missing value:", unknown_percent, "%")

# Kiểm tra và cập nhật missing_dict
print("\nSố lượng missing value sau khi điền: ")
for col in ['store_name']:
    number_missing_value = df[col].isna().sum()
    missing_dict[col] = number_missing_value
    
    print(f"{col}: {missing_dict[col]}")      

# Đối với cancel_by_seller_rate_status nếu cancel_by_seller_rate tồn tại thì điền mode
# Nếu không tồn tại thì điền Unknown
print("Số lượng missing value trước khi điền: ")
for col in ['cancel_by_seller_rate_status']:
    print(f"{col}: {missing_dict[col]}")

# Lấy giá trị mode
mode_status = df['cancel_by_seller_rate_status'].mode()
# Lấy giá trị mode đầu tiên
if len(mode_status) > 0:
    mode_status = mode_status.iloc[0]
else:
    mode_status = "Unknown"

# Điền giá trị
df['cancel_by_seller_rate_status'] = df.apply(
    lambda row: mode_status if pd.notnull(row['cancel_by_seller_rate']) and pd.isnull(row['cancel_by_seller_rate_status'])
                else ("Unknown" if pd.isnull(row['cancel_by_seller_rate']) and pd.isnull(row['cancel_by_seller_rate_status'])
                      else row['cancel_by_seller_rate_status']),
    axis=1
)
# Kiểm tra và cập nhật missing_dict
print("Số lượng missing value sau khi điền: ")
for col in ['cancel_by_seller_rate_status']:
    number_missing_value = df[col].isna().sum()
    missing_dict[col] = number_missing_value
    
    print(f"{col}: {missing_dict[col]}") 

# Đối với return_rate_status nếu return_rate tồn tại thì điền mode
# Nếu không tồn tại thì điền Unknown
 
print("Số lượng missing value trước khi điền: ")
for col in ['return_rate_status']:
    print(f"{col}: {missing_dict[col]}")

# Lấy giá trị mode
mode_status = df['return_rate_status'].mode()
# Lấy giá trị mode đầu tiên
if len(mode_status) > 0:
    mode_status = mode_status.iloc[0]
else:
    mode_status = "Unknown"

# Điền giá trị
df['return_rate_status'] = df.apply(
    lambda row: mode_status if pd.notnull(row['return_rate']) and pd.isnull(row['return_rate_status'])
                else ("Unknown" if pd.isnull(row['return_rate']) and pd.isnull(row['return_rate_status'])
                      else row['return_rate_status']),
    axis=1
)
# Kiểm tra và cập nhật missing_dict
print("Số lượng missing value sau khi điền: ")
for col in ['return_rate_status']:
    number_missing_value = df[col].isna().sum()
    missing_dict[col] = number_missing_value
    
    print(f"{col}: {missing_dict[col]}")  

output_file_name = "data_final_imputed.csv" 
df.to_csv(output_file_name, index=False)
print(f"\nĐã lưu DataFrame đã xử lý vào file: {output_file_name}")

**Thống kê sau khi điền Missing Value**

In [ ]:
# Thống kê lại sau khi điền
missing_dict = {}

count = 0
for col in text_cols:
    number_missing_value = df[col].isnull().sum()
    if(number_missing_value>0): 
        count+=1
    missing_dict.update({col:number_missing_value})

print(f"{'Cột':30} {'Số lượng':10} {'Phần trăm':10}")
print("-" * 55)

for col, val in missing_dict.items():
    percent = round(val / len(df) * 100, 3)
    print(f"{col:30} {val:<10} {percent:<10}%")

print("\nSố cột chứa missing value: ", count)

### **2.1.3. Xử lý Outliers**

**Xử lý Outliers bằng Winsorizing và Log Transformation**

In [ ]:
# ================== ĐỌC FILE INPUT ==================
input_file = "data_final_imputed.csv"
df = pd.read_csv(input_file)

# ================== HÀM WINSORIZING ==================
def winsorize_series(s, lower_q=0.01, upper_q=0.99):
    lower = s.quantile(lower_q)
    upper = s.quantile(upper_q)
    return s.clip(lower, upper)

# ================== CÁC CỘT CẦN XỬ LÝ ==================
winsor_log_cols = [
    "price", "original_price", "quantity_sold",
    "review_count", "video_count", "store_review_count",
    "total_follower"
]

winsor_only_cols = ["discount_rate", "image_count"]

df_pre = df.copy()

# ================== XỬ LÝ WINSOR + LOG  ==================
for col in winsor_log_cols:
    if col in df_pre.columns:
        numeric_col = pd.to_numeric(df_pre[col], errors="coerce")
        # Winsorize
        numeric_col = winsorize_series(numeric_col)
        # Clip lower = 0 để đảm bảo không âm khi lấy log
        numeric_col = numeric_col.clip(lower=0)
        # Áp dụng log1p và gán lại cột
        df_pre[col] = np.log1p(numeric_col)

# ================== WINSOR NHẸ ==================
for col in winsor_only_cols:
    if col in df_pre.columns:
        numeric_col = pd.to_numeric(df_pre[col], errors="coerce")
        df_pre[col] = winsorize_series(numeric_col)

# ================== LƯU FILE OUTPUT ==================
output_file = "data_outliers_removed.csv"
df_pre.to_csv(output_file, index=False, encoding="utf-8")
print(f"\n Đã lưu file tại: {output_file}")

**Thống kê sau khi xử lý Outliers**

In [ ]:
# ================== THỐNG KÊ ==================
selected_cols = [
    'price', 'original_price', 'discount_rate', 'quantity_sold',
    'rating_average', 'review_count', 'is_return_policy',
    'is_freeship_xtra', 'is_authentic', 'image_count', 'video_count',
    'is_brand', 'store_review_count', 'total_follower', 'is_official',
    'cancel_by_seller_rate', 'return_rate'
]

selected_numeric_cols = [c for c in selected_cols if c in df_pre.columns]

stats = []

for col in selected_numeric_cols:
    s = pd.to_numeric(df_pre[col], errors="coerce").dropna()
    
    stats.append({
        "Cột": col,
        "Min": s.min(),
        "Max": s.max(),
        "Mean": s.mean(),
        "Median": s.median(),
        "Std": s.std(),
        "Variance": s.var(),
        "Q1 (25%)": s.quantile(0.25),
        "Q3 (75%)": s.quantile(0.75)
    })

stats_df = pd.DataFrame(stats)

print("\n==================== THỐNG KÊ SAU TIỀN XỬ LÝ ====================")
print(stats_df.to_string(index=False))

# ================== VẼ HISTOGRAM ==================
plt.style.use("ggplot")

for col in selected_numeric_cols:
    plt.figure(figsize=(7, 4))
    plt.hist(df_pre[col].dropna(), bins=40, color="skyblue", edgecolor="black")
    plt.title(f"Histogram của {col}")
    plt.xlabel(col)
    plt.ylabel("Tần suất")
    plt.tight_layout()
    plt.show()

### **2.1.4. Xử lý trùng lặp**

In [150]:
# Không trùng 

### **2.1.5. Làm sạch text (chuẩn hóa text)**

**Hàm chuyển chuỗi về chữ thường, loại bỏ khoảng trắng thừa và kí tự đặc biệt**

In [151]:
df = pd.read_csv("data_outliers_removed.csv")

def normalize_text(s):
    if not isinstance(s, str):
        return ""
    s = unicodedata.normalize("NFC", s)
    s = s.replace("...", " ")
    s = re.sub(r"\(.*?\)", "", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip().lower()

**Hàm chuẩn hóa `origin`**

In [ ]:
# Chuẩn hóa xuất xứ VN, VietNam, Việt Nam,...
def clean_origin(origin):
    if not isinstance(origin, str) or origin.strip() == "":
        return "Other"

    text = normalize_text(origin)

    # Tách theo nhiều dạng dấu phân cách
    parts = re.split(r"[/,;|]", text)
    parts = [p.strip() for p in parts if p.strip()]

    # Từ điển chuẩn hóa về tên quốc gia tiếng Anh
    mapping = {
        # Việt Nam
        "việt nam": "Vietnam", "viet nam": "Vietnam", "vietnam": "Vietnam",

        # Trung Quốc
        "trung quốc": "China", "trung quoc": "China", "china": "China",
        "tq": "China", "hn/tq": "China", "hk/tq": "China",

        # Nhật Bản
        "nhật bản": "Japan", "nhat ban": "Japan", "japan": "Japan",

        # Hàn Quốc
        "hàn quốc": "South Korea", "han quoc": "South Korea",
        "korea": "South Korea", "south korea": "South Korea",

        # Mỹ
        "mỹ": "USA", "my": "USA", "usa": "USA",
        "united states": "USA", "us": "USA",

        # Hong Kong
        "hong kong": "Hong Kong", "hồng kông": "Hong Kong", "hk": "Hong Kong",

        # Đài Loan
        "đài loan": "Taiwan", "dai loan": "Taiwan", "taiwan": "Taiwan",

        # Ấn Độ
        "ấn độ": "India", "an do": "India", "india": "India",

        # Thái Lan
        "thái lan": "Thailand", "thai lan": "Thailand", "thailand": "Thailand",

        # Đức
        "đức": "Germany", "duc": "Germany", "germany": "Germany",

        # Anh
        "anh": "UK", "england": "UK", "uk": "UK",

        # Pháp
        "pháp": "France", "phap": "France", "france": "France",

        # Úc
        "úc": "Australia", "uc": "Australia", "australia": "Australia",

        # Italy
        "ý": "Italy", "y": "Italy", "italia": "Italy", "italy": "Italy",

        # Malaysia
        "malaysia": "Malaysia", "mã lai": "Malaysia",

        # Philippines
        "philippines": "Philippines", "philipin": "Philippines",
        "philipines": "Philippines", "philipin": "Philippines",

        # Singapore
        "singapore": "Singapore",

        # Hà Lan
        "hà lan": "Netherlands", "ha lan": "Netherlands",

        # Thổ Nhĩ Kỳ
        "thổ nhĩ kì": "Turkey", "thổ nhĩ kỳ": "Turkey",
        "tho nhi ky": "Turkey", "turkey": "Turkey",

        # Pakistan
        "pakistan": "Pakistan",

        # Iran
        "iran": "Iran",

        # Myanmar
        "myanmar": "Myanmar",

        # Cambodia
        "campuchia": "Cambodia", "cam pu chia": "Cambodia", "cambodia": "Cambodia",

        # Ai Cập
        "ai cập": "Egypt", "ai cap": "Egypt", "egypt": "Egypt",

        # Nga
        "nga": "Russia", "russia": "Russia",

        # Canada
        "canada": "Canada",

        # Ba Lan
        "ba lan": "Poland", "poland": "Poland",

        # Hungary
        "hungary": "Hungary",

        # Cộng Hòa Séc
        "cộng hòa séc": "Czech Republic", "sec": "Czech Republic",
        "czech": "Czech Republic",

        # Bỉ
        "bỉ": "Belgium", "bi": "Belgium", "belgium": "Belgium",

        # Israel
        "israel": "Israel",

        # Chile
        "chile": "Chile",

        # Mexico
        "mexico": "Mexico",

        # Ireland
        "ireland": "Ireland",

        # Nam Phi
        "nam phi": "South Africa", "south africa": "South Africa",

        # Ukraine
        "ukraine": "Ukraine",

        # Tunisia
        "tunisia": "Tunisia",

        # Cuba
        "cuba": "Cuba",

        # Sri Lanka
        "sri lanka": "Sri Lanka",

        # Brazil
        "brazil": "Brazil",

        # Madagascar
        "madagascar": "Madagascar",

        # Đan Mạch
        "đan mạch": "Denmark", "dan mach": "Denmark",

        # Hy Lạp
        "hy lạp": "Greece", "hy lap": "Greece", "greece": "Greece",

        # Thụy Điển
        "thụy điển": "Sweden", "thuy dien": "Sweden", "sweden": "Sweden",

        # Thụy Sĩ
        "thụy sỹ": "Switzerland", "thuy sy": "Switzerland",
        "switzerland": "Switzerland",

        # United Kingdom, Scotland
        "scotland": "UK",

        # New Zealand
        "new zealand": "New Zealand",

        # Slovenia
        "slovenia": "Slovenia",

        # Áo
        "áo": "Austria", "ao": "Austria", "austria": "Austria",

        # Finland
        "phần lan": "Finland", "phan lan": "Finland", "finland": "Finland",

        # Nhiều giá trị đặc biệt
        "nhiều quốc gia": "Other",
        "khác": "Other",
        "unknown": "Other",
        "không xác định": "Other",
    }

    result = set()

    for p in parts:
        if p in mapping:
            result.add(mapping[p])
            continue
        # Không nhận diện được → Other
        result.add("Other")

    if len(result) == 0:
        return "Other"

    # Sắp xếp cho đẹp
    return ", ".join(sorted(result))

df['origin'] = df['origin'].apply(clean_origin)
print(df['origin'].unique())

## **2.2. Data Reduction**

Phân loại cột theo dạng: Numeric, ID, Binary, Text
- Các cột **numeric**: drop cột `return_rate` và `cancel_by_seller_rate` do không có giá trị dự đoán.
- Các cột **text**:
    + Drop cột `cancel_by_seller_rate_status` và `return_rate_status` do chỉ có 1 giá trị dự đoán.
    + Drop cột `product_url` do chỉ mang tính chất phân biệt sản phẩm, không có giá trị dự đoán
    + Drop cột `store_name`,`brand_name` ,`category_name` và chỉ giữ lại `category_root_name` để tránh chi tiết hóa và tránh model học vẹt.  
- Các cột **ID**:
    + Drop tất cả các cột bao gồm `product_id`, `category_id` và `store_id` do ID chỉ dùng để định danh.
- Các cột **binary**: giữ nguyên

In [ ]:
drop_cols = ['cancel_by_seller_rate', 'return_rate',
             'cancel_by_seller_rate_status', 'return_rate_status', 'product_url',
             'category_name', 'store_name', 'brand_name',
             'product_id', 'store_id', 'category_id']
df.drop(columns=drop_cols, inplace=True)

# In ra 5 dòng ngẫu nhiên để xem data đã thay đổi 
print(df.sample(5))

## **2.3. Data Transformation**

### **2.3.1. Feature Engineering**

- `reputation_score`: điểm uy tín thực sự giữa `rating_average` và `review_count` (Rating 5 sao mà chỉ có 1 review thì không bằng 4.5 sao mà có 1000 review.)
- `trust_level`: điểm tín nhiệm là tổng giữa các flag uy tín `is_official`, `is_authentic` và `is_brand`
- `review_to_sold_ratio`: tỉ lệ tương tác so với lượt mua (`review_count`/(`quantity_sold` + 1))
- `shop_potential`: đo độ nổi tiếng của shop dựa trên tổng của `store_review_count` và `total_follower`  

**Phục vụ cho câu hỏi 2**
- `total_visuals`: tổng số lượng video và hình ảnh
- `has_video`: flag kiểm tra sản phẩm có video không  

**Phục vụ cho câu hỏi 3**
- `discount_amount`: số tiền giảm thực tế (hiệu của `original_price` và `price`)
- `price_vs_category`: giá tương đối của sản phẩm so với danh mục, so sánh giá sản phẩm so với giá trung bình của tất cả các sản phẩm cùng danh mục. <1: rẻ hơn trung bình; >1: mắc hơn trung bình.  

**Phục vụ cho câu hỏi 5**
- `hot_keyword_count`: đếm số lần xuất hiện những từ khóa "giật tít" có trong tiêu đề.  

**Phục vụ cho câu hỏi 7**
- `name_length`: độ dài của tiêu đề
- `name_word_count`: số chữ của tiêu đề

In [ ]:
print("TẠO CÁC FEATURE MỚI")

# Reputation Score 
# Công thức: Rating * log(Review + 1).
df['reputation_score'] = df['rating_average'] * np.log1p(df['review_count'])

# Trust Level (Điểm tín nhiệm)
df['trust_level'] = (df['is_official'] + df['is_authentic'] + 
                     df['is_brand'])

# Review to Sold Ratio (Tỷ lệ tương tác)
# Công thức: Review / (Sold + 1)
df['review_to_sold_ratio'] = df['review_count'] / (df['quantity_sold'].fillna(0) + 1)

# Shop Potential (Độ nổi tiếng của Shop)
# Công thức: Store Review + Total Follower
df['shop_potential'] = df['store_review_count'].fillna(0) + df['total_follower'].fillna(0)

# Total Visuals (Tổng tư liệu hình ảnh)
df['total_visuals'] = df['image_count'].fillna(0) + df['video_count'].fillna(0)

# Has Video (Có video không?)
df['has_video'] = df['video_count'].apply(lambda x: 1 if x > 0 else 0)

# Discount Amount (Số tiền giảm thực tế)
df['discount_amount'] = df['original_price'] - df['price']
# Xử lý số âm (nếu có lỗi dữ liệu)
df['discount_amount'] = df['discount_amount'].apply(lambda x: x if x > 0 else 0)

#Price vs Category (Giá tương đối)
# So sánh giá sp với giá trung bình của danh mục đó
if 'category_root_name' in df.columns:
    # Tính giá trung bình của từng danh mục
    df['category_avg_price'] = df.groupby('category_root_name')['price'].transform('mean')
    # Tính tỷ lệ: Giá SP / (Giá TB + 1)
    df['price_vs_category'] = df['price'] / (df['category_avg_price'] + 1)
else:
    df['price_vs_category'] = 1.0 # Mặc định nếu thiếu cột category

# Drop cột category_avg_price sau khi dùng xong
df.drop(columns='category_avg_price', inplace=True, errors='ignore')

# Hot Keyword Count (Đếm từ khóa "giật tít")
# Danh sách các từ khóa hấp dẫn
keywords = ['chính hãng', 'cao cấp', 'freeship', 'xịn', 'hot', 'giảm', 'sale', 'tặng', 'combo', 'bảo hành', 'nhập khẩu', 'siêu rẻ']
pattern = '|'.join(keywords)

# Đếm số lần xuất hiện (count)
df['hot_keyword_count'] = df['product_name'].astype(str).str.lower().str.count(pattern).fillna(0).astype(int)

# Name Length (Độ dài tiêu đề - Ký tự)
df['name_length'] = df['product_name'].astype(str).apply(len)

# Name Word Count (Độ dài tiêu đề - Số từ)
df['name_word_count'] = df['product_name'].astype(str).apply(lambda x: len(x.split()))

# Drop cột product_name sau khi dùng xong
df.drop(columns='product_name', inplace=True, errors='ignore')

print("Đã tạo xong 11 feature mới theo yêu cầu.")

# Hiển thị 5 dòng đầu của các cột mới để kiểm tra
new_cols = ['reputation_score', 'trust_level', 'shop_potential', 
            'has_video', 'discount_amount', 'price_vs_category',
            'hot_keyword_count', 'name_word_count']
            
print(df[new_cols].head())

### **2.3.2. Chuẩn hóa dữ liệu**

**One-hot encoding cho `category_root_name`**

In [ ]:
if 'category_root_name' in df.columns:
    df = pd.get_dummies(df, columns=['category_root_name'], drop_first=True, dtype=int)
    print("One-hot encoding cho category_root_name hoàn tất")

# Drop cột category_avg_price sau khi dùng xong
df.drop(columns='category_avg_price', inplace=True, errors='ignore')

**One-hot encoding cho `origin`**

In [156]:
# Lấy giá trị để one-hot encoding
def split_origin(s):
    """
    Tách tất cả các giá trị trong cột origin thành list
    """
    if not isinstance(s, str):
        return []
    # Tách theo dấu phẩy, '/', '-', giữ các giá trị quốc gia
    parts = re.split(r'[,/\-]', s)
    # Xóa khoảng trắng thừa
    parts = [p.strip() for p in parts if p.strip()]
    return parts

# Tạo cột mới dạng list
df['origin_list'] = df['origin'].apply(split_origin)

# One-hot encoding
mlb = MultiLabelBinarizer()
origin_encoded = pd.DataFrame(mlb.fit_transform(df['origin_list']),
                              columns=mlb.classes_,
                              index=df.index)

# Lưu lại các cột
origin_cols = origin_encoded.columns.tolist()
# Nối vào DataFrame gốc
df = pd.concat([df, origin_encoded], axis=1)

# Xóa cột tạm
df.drop(columns=['origin_list'], inplace=True)

df.head()

# Tính tổng từng cột one-hot
origin_per = (df[origin_cols].sum() / len(df) * 100)

# Cột giữ lại (> 2%)
keep_origin = origin_per[origin_per > 2].index.tolist()
# Cột gộp vào Others (<= 2%)
other_origin = origin_per[origin_per <= 2].index.tolist()

# Tạo cột Others bằng tổng các cột nhỏ
df['Others'] = df[other_origin].sum(axis=1) + df['Other']

# Xóa các cột nhỏ
df.drop(columns=['Other'], inplace=True)
df.drop(columns=other_origin, inplace=True)

### **2.3.3. Scaling**

Cần chia tập dữ liệu thành train và test trước khi scaling.

In [157]:
# Chia tập train-test
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# Min-max Scaling cho các cột numeric tập train
# Không dùng thư viện
import numpy as np
from sklearn.preprocessing import MinMaxScaler
    
# Loại bỏ các cột Binary (is_official, is_authentic, has_video, v.v.)
cols_to_scale = [col for col in train_df.columns 
                 if train_df[col].dtype in ['float64', 'int64'] 
                 and col not in ['is_official', 'is_authentic', 'is_brand', 
                                 'is_freeship_xtra', 'is_return_policy', 'has_video']]

# 3. Xử lý NaN và inf
train_df[cols_to_scale] = train_df[cols_to_scale].replace([np.inf, -np.inf], 0).fillna(0)

# 4. Áp dụng MinMaxScaler (0-1) cho tất cả các cột liên tục (đã log hoặc chưa)
scaler = MinMaxScaler()
train_df[cols_to_scale] = scaler.fit_transform(train_df[cols_to_scale])

# Lưu file train-test và tập full
train_df.to_csv('train_data_final.csv', index=False)
test_df.to_csv('test_data_final.csv', index=False)
df.to_csv('full_data_final.csv', index=False)